In [1]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time

options = Options()
# 파트너센터는 직접 로그인을 해야 하므로 headless(화면 숨김) 모드를 사용하지 않습니다.
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

def crawl_partner_reviews(url):
    driver.get(url)

    # 1. 수동 로그인 대기 (가장 핵심!)
    print("\n" + "="*50)
    print("브라우저가 열렸습니다. 직접 로그인을 진행해 주세요.")
    print("로그인 후, 크롤링할 [리뷰 목록 페이지]까지 직접 이동해 주세요.")
    print("="*50 + "\n")

    # 사용자가 터미널에서 엔터를 칠 때까지 파이썬이 여기서 대기합니다.
    input("👉 리뷰 페이지 로딩이 완벽히 끝났다면, 여기(터미널)를 클릭하고 [Enter] 키를 누르세요!")

    review_data = []
    page_count = 1

    while True:
        print(f"--- {page_count}페이지 데이터 수집 중 ---")

        # 2. 리뷰 컨테이너 로딩 대기
        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "div.css-8nuvr6"))
            )
        except Exception as e:
            print("리뷰 항목을 찾을 수 없습니다. 페이지를 확인해 주세요.")
            break

        # 3. 데이터 추출
        items = driver.find_elements(By.CSS_SELECTOR, "div.css-8nuvr6")

        for item in items:
            try:
                # [객실명] p.css-1w7m3on
                room_elems = item.find_elements(By.CSS_SELECTOR, "p.css-1w7m3on")
                room_type = room_elems[0].text if room_elems else "객실명 없음"

                # [리뷰내용] p.css-5t5ayp
                content_elems = item.find_elements(By.CSS_SELECTOR, "p.css-5t5ayp")
                content = content_elems[0].get_attribute("textContent").strip() if content_elems else "내용 없음"

                # [아이디 & 작성일자] 같은 클래스(span.css-yzqowo)를 공유하므로 리스트로 받아옴
                # 사진을 보면 1번째가 아이디, 2번째가 작성일자입니다.
                info_spans = item.find_elements(By.CSS_SELECTOR, "div.css-1mah1oy span.css-yzqowo")

                if len(info_spans) >= 2:
                    user_id = info_spans[0].text
                    date = info_spans[1].text
                else:
                    user_id = "아이디 없음"
                    date = "날짜 없음"

                review_data.append({
                    "작성일자": date,
                    "아이디": user_id,
                    "객실명": room_type,
                    "리뷰내용": content
                })
            except Exception as e:
                continue

        # 4. [다음] 페이지 이동 로직 (페이지네이션)
        try:
            # 사진에서 확인된 페이지네이션 영역(div.css-2y6e0k) 안의 마지막 버튼이 '다음' 버튼임
            next_btn = driver.find_element(By.CSS_SELECTOR, "div.css-2y6e0k button:last-child")

            # 다음 버튼이 비활성화(disabled) 상태라면 마지막 페이지
            if next_btn.get_attribute("disabled"):
                print("마지막 페이지에 도달했습니다. 수집을 종료합니다.")
                break

            # 자바스크립트로 강제 클릭 후 대기
            driver.execute_script("arguments[0].click();", next_btn)
            time.sleep(2)
            page_count += 1

        except Exception as e:
            print("페이지 이동 버튼을 찾는 데 실패했습니다. 수집을 종료합니다.")
            break

    return review_data

try:
    # 파트너센터 로그인 URL (또는 즐겨찾기 해둔 리뷰 페이지 URL)
    url = "https://partner.goodchoice.kr/reviews/review-all-list"

    results = crawl_partner_reviews(url)

    if len(results) > 0:
        df = pd.DataFrame(results)
        df.to_excel("yeogi_reviews.xlsx", index=False)
        print(f"\n🎉 성공! 총 {len(df)}개의 리뷰가 'yeogi_reviews.xlsx'에 저장되었습니다.")

        # 첫 번째 데이터 살짝 보여주기
        print("첫 번째 데이터 확인:", results[0])
    else:
        print("수집된 데이터가 없습니다.")

finally:
    driver.quit()


브라우저가 열렸습니다. 직접 로그인을 진행해 주세요.
로그인 후, 크롤링할 [리뷰 목록 페이지]까지 직접 이동해 주세요.

--- 1페이지 데이터 수집 중 ---
--- 2페이지 데이터 수집 중 ---
--- 3페이지 데이터 수집 중 ---
--- 4페이지 데이터 수집 중 ---
--- 5페이지 데이터 수집 중 ---
--- 6페이지 데이터 수집 중 ---
--- 7페이지 데이터 수집 중 ---
--- 8페이지 데이터 수집 중 ---
--- 9페이지 데이터 수집 중 ---
--- 10페이지 데이터 수집 중 ---
--- 11페이지 데이터 수집 중 ---
--- 12페이지 데이터 수집 중 ---
--- 13페이지 데이터 수집 중 ---
--- 14페이지 데이터 수집 중 ---
--- 15페이지 데이터 수집 중 ---
--- 16페이지 데이터 수집 중 ---
--- 17페이지 데이터 수집 중 ---
--- 18페이지 데이터 수집 중 ---
--- 19페이지 데이터 수집 중 ---
--- 20페이지 데이터 수집 중 ---
--- 21페이지 데이터 수집 중 ---
--- 22페이지 데이터 수집 중 ---
--- 23페이지 데이터 수집 중 ---
--- 24페이지 데이터 수집 중 ---
--- 25페이지 데이터 수집 중 ---
--- 26페이지 데이터 수집 중 ---
--- 27페이지 데이터 수집 중 ---
--- 28페이지 데이터 수집 중 ---
--- 29페이지 데이터 수집 중 ---
--- 30페이지 데이터 수집 중 ---
마지막 페이지에 도달했습니다. 수집을 종료합니다.

🎉 성공! 총 296개의 리뷰가 'yeogi_partner_reviews.xlsx'에 저장되었습니다.
첫 번째 데이터 확인: {'작성일자': '2026.02.18', '아이디': '잠자리예민', '객실명': '글램핑 5호(모던카페룸)', '리뷰내용': '글램핑장 예약해서 잘 쉬다가 갑니다.\n안에 간편하게 이것저것 구매할수 있는 매점도 있고 보드게임이나 필요한 식기류 등도 잘 구비되어 있어 